# FedKG-Maint — Phase 4: Factory-Local Maintenance Knowledge Graphs v1

**Purpose:** Build one maintenance knowledge graph per factory from the Azure Predictive Maintenance data, using one shared schema/ontology. This notebook reads the existing Phase 1 factory assignment; it does not invent a new machine-to-factory partition.

**Inputs:** `data/raw/PdM_{telemetry,errors,maint,failures,machines}.csv` and `prep_v3/fedkg_data.py` under `MyDrive/FedKG-Maint`.

**Outputs:** shared ontology (Turtle + JSON), five factory-local RDF graphs (Turtle), normalized evidence tables, graph statistics, and a validation report under `phase4_local_knowledge_graphs_v1/`.

**Important boundaries:**
- Azure-only, per project decision; NASA C-MAPSS is not used here.
- A graph edge records an observed association, not causal proof. This ontology deliberately does not assert that an error or maintenance action *causes* a failure.
- The raw data are centrally available in this Colab workflow. Separate graph files and per-factory scalers support the intended logical separation, but do **not** prove physical data isolation. A real deployment would run graph creation inside each factory.
- Event timestamps are retained. Downstream explanation retrieval must enforce an `as_of` cutoff so facts after a prediction timestamp are never returned as evidence.

In [ ]:
from pathlib import Path
import os, sys, json, math, re, warnings, subprocess, importlib.util
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import rdflib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdflib"])

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

RAW_DIR = PROJECT_DIR / "data" / "raw"
PREP_DIR = PROJECT_DIR / "prep_v3"
OUT_DIR = PROJECT_DIR / "phase4_local_knowledge_graphs_v1"
for name in ["ontology", "graphs", "tables", "reports", "logs"]:
    (OUT_DIR / name).mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)
print("Output:", OUT_DIR)
assert RAW_DIR.exists(), f"Raw data folder not found: {RAW_DIR}"
assert (PREP_DIR / "fedkg_data.py").exists(), "Phase 1 prep_v3/fedkg_data.py is required to preserve the existing factory assignment."

In [ ]:
def locate_csv(*candidate_names):
    files = list(RAW_DIR.rglob("*.csv"))
    lookup = {p.name.casefold(): p for p in files}
    for name in candidate_names:
        if name.casefold() in lookup:
            return lookup[name.casefold()]
    wanted = {n.casefold() for n in candidate_names}
    found = [p for p in files if p.stem.casefold() in {Path(n).stem.casefold() for n in candidate_names}]
    if found:
        return found[0]
    raise FileNotFoundError(f"Could not find one of {candidate_names} under {RAW_DIR}. Available CSVs: {[p.name for p in files[:30]]}")

PATHS = {
    "telemetry": locate_csv("PdM_telemetry.csv", "telemetry.csv"),
    "errors": locate_csv("PdM_errors.csv", "errors.csv"),
    "maintenance": locate_csv("PdM_maint.csv", "PdM_maintenance.csv", "maintenance.csv", "maint.csv"),
    "failures": locate_csv("PdM_failures.csv", "failures.csv"),
    "machines": locate_csv("PdM_machines.csv", "machines.csv"),
}
print("Resolved inputs:")
for k, v in PATHS.items(): print(f"  {k:12s} {v}")

def normalize_id(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    try:
        return str(int(float(s)))
    except Exception:
        return s

def pick_column(df, candidates, required=True, table_name="table"):
    cols = {str(c).strip().casefold(): c for c in df.columns}
    for c in candidates:
        if c.casefold() in cols:
            return cols[c.casefold()]
    if required:
        raise ValueError(f"{table_name}: expected one of columns {candidates}; found {list(df.columns)}")
    return None

def load_source(name):
    df = pd.read_csv(PATHS[name], low_memory=False)
    machine_col = pick_column(df, ["machineID", "machine_id"], table_name=name)
    if machine_col != "machineID": df = df.rename(columns={machine_col: "machineID"})
    df["machineID"] = df["machineID"].map(normalize_id)
    dt_col = pick_column(df, ["datetime", "timestamp", "date"], required=(name != "machines"), table_name=name)
    if dt_col and dt_col != "datetime": df = df.rename(columns={dt_col: "datetime"})
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
        if df["datetime"].isna().any():
            bad = int(df["datetime"].isna().sum())
            raise ValueError(f"{name}: {bad} invalid datetime values; resolve before KG creation.")
    return df

telemetry = load_source("telemetry")
errors = load_source("errors")
maintenance = load_source("maintenance")
failures = load_source("failures")
machines = load_source("machines")

SENSORS = ["volt", "rotate", "pressure", "vibration"]
missing_sensors = [c for c in SENSORS if c not in telemetry.columns]
if missing_sensors:
    raise ValueError(f"Telemetry is missing expected Azure sensor columns: {missing_sensors}; found {list(telemetry.columns)}")
for c in SENSORS:
    telemetry[c] = pd.to_numeric(telemetry[c], errors="coerce")
if telemetry[SENSORS].isna().any().any():
    raise ValueError("Sensor columns contain missing or non-numeric values; stop rather than silently inventing values.")

error_code_col = pick_column(errors, ["errorID", "error_id", "errorCode", "error_code"], table_name="errors")
if error_code_col != "errorID": errors = errors.rename(columns={error_code_col: "errorID"})
maintenance_comp_col = pick_column(maintenance, ["comp", "component", "componentID", "component_id"], table_name="maintenance")
if maintenance_comp_col != "comp": maintenance = maintenance.rename(columns={maintenance_comp_col: "comp"})
failure_comp_col = pick_column(failures, ["failure", "comp", "component", "componentID", "component_id", "failureType"], required=False, table_name="failures")
if failure_comp_col is not None and failure_comp_col != "failure_type":
    failures = failures.rename(columns={failure_comp_col: "failure_type"})
else:
    failures["failure_type"] = "unknown_component"

model_col = pick_column(machines, ["model", "machineModel", "machine_model"], required=False, table_name="machines")
age_col = pick_column(machines, ["age", "machineAge", "machine_age"], required=False, table_name="machines")
if model_col and model_col != "model": machines = machines.rename(columns={model_col: "model"})
if age_col and age_col != "age": machines = machines.rename(columns={age_col: "age"})
if "model" not in machines.columns: machines["model"] = "unknown_model"
if "age" not in machines.columns: machines["age"] = np.nan
machines["model"] = machines["model"].fillna("unknown_model").astype(str)
machines["age"] = pd.to_numeric(machines["age"], errors="coerce")

print({"telemetry_rows": len(telemetry), "error_rows": len(errors), "maintenance_rows": len(maintenance),
       "failure_rows": len(failures), "machine_rows": len(machines), "sensor_features": SENSORS})

In [ ]:
# Reuse the authoritative client mapping from Phase 1 metadata; do not re-partition machines here.
sys.path.insert(0, str(PREP_DIR))
import importlib
sys.modules.pop("fedkg_data", None)
importlib.invalidate_caches()
import fedkg_data

mapping_parts = []
for split_name in ["train", "validation", "test_site_full"]:
    _, _, meta = fedkg_data.load_split(split_name, policy="local", feature_set="full")
    needed = {"machineID", "client_id"}
    if not needed.issubset(meta.columns):
        raise ValueError(f"Phase 1 metadata for {split_name} lacks {needed - set(meta.columns)}; found {list(meta.columns)}")
    part = meta[["machineID", "client_id"]].copy()
    part["machineID"] = part["machineID"].map(normalize_id)
    part["client_id"] = part["client_id"].astype(str)
    mapping_parts.append(part.drop_duplicates())

mapping_all = pd.concat(mapping_parts, ignore_index=True).drop_duplicates()
ids_per_machine = mapping_all.groupby("machineID")["client_id"].nunique()
conflicting = ids_per_machine[ids_per_machine != 1]
if len(conflicting):
    raise AssertionError(f"Machines mapped to multiple factories: {conflicting.to_dict()}")
factory_map = mapping_all.drop_duplicates("machineID").set_index("machineID")["client_id"].to_dict()
EXPECTED_FACTORIES = [f"factory_{i}" for i in range(1, 6)]
if sorted(set(factory_map.values())) != EXPECTED_FACTORIES:
    raise AssertionError(f"Expected clients {EXPECTED_FACTORIES}; observed {sorted(set(factory_map.values()))}")

all_source_machines = set(machines["machineID"].dropna()) | set(telemetry["machineID"].dropna())
missing_map = all_source_machines - set(factory_map)
if missing_map:
    raise AssertionError(f"Some source machines have no Phase 1 factory mapping: {sorted(missing_map)[:20]}")

mapping_df = pd.DataFrame([{"machineID": m, "client_id": factory_map[m]} for m in sorted(factory_map, key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else x))])
mapping_df.to_csv(OUT_DIR / "tables" / "machine_factory_mapping.csv", index=False)
print("Factory assignment preserved from Phase 1:")
print(mapping_df.groupby("client_id").agg(machines=("machineID", "nunique")).to_string())
assert mapping_df["machineID"].nunique() == len(mapping_df), "Each machine must map to exactly one factory."

In [ ]:
# Normalize event tables and retain the source's observed values only.
for df_name, df in [("errors", errors), ("maintenance", maintenance), ("failures", failures)]:
    if df["datetime"].isna().any():
        raise ValueError(f"{df_name}: invalid datetimes detected")
    unknown_machines = set(df["machineID"].dropna()) - set(factory_map)
    if unknown_machines:
        raise AssertionError(f"{df_name}: event rows refer to unmapped machines {sorted(unknown_machines)[:10]}")
    df["client_id"] = df["machineID"].map(factory_map)

errors["errorID"] = errors["errorID"].fillna("unknown_error").astype(str)
maintenance["comp"] = maintenance["comp"].fillna("unknown_component").astype(str)
failures["failure_type"] = failures["failure_type"].fillna("unknown_component").astype(str)
errors = errors.drop_duplicates(subset=["machineID", "datetime", "errorID"]).copy()
maintenance = maintenance.drop_duplicates(subset=["machineID", "datetime", "comp"]).copy()
failures = failures.drop_duplicates(subset=["machineID", "datetime", "failure_type"]).copy()

# One telemetry node per machine-day. It retains descriptive statistics, not every raw row.
telemetry["client_id"] = telemetry["machineID"].map(factory_map)
telemetry["day"] = telemetry["datetime"].dt.floor("D")
agg = {s: ["mean", "std", "min", "max"] for s in SENSORS}
daily = telemetry.groupby(["machineID", "client_id", "day"], as_index=False).agg(agg)
daily.columns = ["machineID", "client_id", "day"] + [f"{s}_{stat}" for s in SENSORS for stat in ["mean", "std", "min", "max"]]
daily = daily.merge(telemetry.groupby(["machineID", "client_id", "day"]).size().rename("n_samples").reset_index(),
                    on=["machineID", "client_id", "day"], how="left")

# Evidence tables are saved for later timestamp-filtered retrieval and audits.
for name, df in [("telemetry_daily_summaries", daily), ("error_events", errors),
                 ("maintenance_events", maintenance), ("failure_events", failures)]:
    to_save = df.copy()
    for c in to_save.columns:
        if pd.api.types.is_datetime64_any_dtype(to_save[c]):
            to_save[c] = to_save[c].map(lambda x: x.isoformat() if pd.notna(x) else "")
    to_save.to_csv(OUT_DIR / "tables" / f"{name}.csv", index=False)

print("Prepared factory-local evidence tables:")
print("daily telemetry summaries:", len(daily))
print("unique error occurrences:", len(errors))
print("unique maintenance events:", len(maintenance))
print("unique failure records:", len(failures))

In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, Literal, URIRef
from urllib.parse import quote

BASE = "https://example.org/fedkg-maint/"
ONT = Namespace(BASE + "ontology#")
RES = Namespace(BASE + "resource/")

CLASSES = {
    "Factory": "A factory partition used as a logical data boundary.",
    "Machine": "An industrial machine from the Azure predictive-maintenance dataset.",
    "MachineModel": "A model category recorded in the machine inventory.",
    "Component": "A component name observed in maintenance or failure records.",
    "ErrorCode": "An error code recorded by the source system.",
    "ErrorOccurrence": "An observed error event with its source timestamp and error code.",
    "MaintenanceEvent": "An observed maintenance record for a machine and component.",
    "FailureEvent": "An observed failure record from the source dataset.",
    "TelemetryDailySummary": "A per-machine, per-day aggregate of measured telemetry.",
}
OBJECT_PROPERTIES = {
    "hasMachine": ("Factory", "Machine", "Factory membership in this local graph."),
    "belongsToFactory": ("Machine", "Factory", "Factory assignment copied from Phase 1 metadata."),
    "usesModel": ("Machine", "MachineModel", "Recorded model category."),
    "hasErrorOccurrence": ("Machine", "ErrorOccurrence", "Observed error event; not a causal assertion."),
    "hasErrorCode": ("ErrorOccurrence", "ErrorCode", "Code recorded for an error occurrence."),
    "hasMaintenanceEvent": ("Machine", "MaintenanceEvent", "Observed maintenance event."),
    "targetsComponent": ("MaintenanceEvent", "Component", "Component named in the maintenance record."),
    "hasFailureEvent": ("Machine", "FailureEvent", "Observed failure event."),
    "failureOfComponent": ("FailureEvent", "Component", "Component label recorded for the failure."),
    "hasDailyTelemetry": ("Machine", "TelemetryDailySummary", "Daily telemetry summary for the machine."),
}
DATA_PROPERTIES = {
    "machineID": "Identifier from the source machine inventory.",
    "modelName": "Machine model category as recorded by the source.",
    "machineAge": "Machine age value from the source inventory.",
    "errorCode": "Error identifier from the source event record.",
    "componentName": "Component label from the source event record.",
    "occurredAt": "Source timestamp for an event.",
    "dailyDate": "Calendar date represented by a telemetry aggregate.",
    "nSamples": "Number of raw telemetry rows in the daily aggregate.",
}
for sensor in SENSORS:
    for stat in ["mean", "std", "min", "max"]:
        DATA_PROPERTIES[f"{sensor}{stat.title()}"] = f"Daily {stat} of the source telemetry field {sensor}."

ontology = Graph()
ontology.bind("fedkg", ONT); ontology.bind("rdf", RDF); ontology.bind("rdfs", RDFS); ontology.bind("owl", OWL); ontology.bind("xsd", XSD)
ontology.add((URIRef(BASE + "ontology"), RDF.type, OWL.Ontology))
ontology.add((URIRef(BASE + "ontology"), RDFS.label, Literal("FedKG-Maint Shared Maintenance Ontology")))
for name, description in CLASSES.items():
    term = ONT[name]
    ontology.add((term, RDF.type, OWL.Class)); ontology.add((term, RDFS.label, Literal(name)))
    ontology.add((term, RDFS.comment, Literal(description)))
for name, (domain, range_, description) in OBJECT_PROPERTIES.items():
    term = ONT[name]
    ontology.add((term, RDF.type, OWL.ObjectProperty)); ontology.add((term, RDFS.label, Literal(name)))
    ontology.add((term, RDFS.domain, ONT[domain])); ontology.add((term, RDFS.range, ONT[range_]))
    ontology.add((term, RDFS.comment, Literal(description)))
for name, description in DATA_PROPERTIES.items():
    term = ONT[name]
    ontology.add((term, RDF.type, OWL.DatatypeProperty)); ontology.add((term, RDFS.label, Literal(name)))
    ontology.add((term, RDFS.comment, Literal(description)))

ontology_path = OUT_DIR / "ontology" / "fedkg_shared_ontology.ttl"
ontology.serialize(destination=str(ontology_path), format="turtle")
ontology_json = {
    "ontology_iri": BASE + "ontology", "namespace": str(ONT),
    "classes": CLASSES,
    "object_properties": {k: {"domain": v[0], "range": v[1], "description": v[2]} for k, v in OBJECT_PROPERTIES.items()},
    "data_properties": DATA_PROPERTIES,
    "causal_assertions": "Not defined. Co-occurrence in these logs must not be described as proven causation.",
    "temporal_rule": "For an explanation at time T, retrieve event and telemetry facts with timestamps <= T only."
}
(OUT_DIR / "ontology" / "fedkg_shared_ontology.json").write_text(json.dumps(ontology_json, indent=2))
print("Shared ontology written:", ontology_path, "| triples:", len(ontology))

In [ ]:
def uri(factory_id, entity_type, key):
    return RES[f"{quote(str(factory_id), safe='')}/{quote(str(entity_type), safe='')}/{quote(str(key), safe='')}"]

def add_label(g, node, label):
    g.add((node, RDFS.label, Literal(str(label))))

def add_dt(g, subject, prop, value, datatype=None):
    if value is None or (isinstance(value, float) and not np.isfinite(value)) or pd.isna(value):
        return
    if datatype is not None:
        lit = Literal(value, datatype=datatype)
    else:
        lit = Literal(value)
    g.add((subject, ONT[prop], lit))

def event_datetime(value):
    ts = pd.Timestamp(value)
    return Literal(ts.isoformat(), datatype=XSD.dateTime)

# Inventory rows can have one row per machine; fail on conflicting duplicate records.
if machines["machineID"].duplicated().any():
    dupe = machines[machines["machineID"].duplicated(keep=False)]["machineID"].unique().tolist()
    raise AssertionError(f"Machine inventory has repeated machine IDs; review source before graph build: {dupe[:20]}")
machine_info = machines.set_index("machineID", drop=False).to_dict("index")

local_graphs = {}
graph_stats = []
for factory_id in EXPECTED_FACTORIES:
    g = Graph()
    g.bind("fedkg", ONT); g.bind("rdf", RDF); g.bind("rdfs", RDFS); g.bind("xsd", XSD)
    factory_node = uri(factory_id, "factory", factory_id)
    g.add((factory_node, RDF.type, ONT.Factory)); add_label(g, factory_node, factory_id)
    factory_machines = sorted([m for m, f in factory_map.items() if f == factory_id], key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else x))

    for mid in factory_machines:
        mnode = uri(factory_id, "machine", mid)
        g.add((mnode, RDF.type, ONT.Machine)); add_label(g, mnode, f"Machine {mid}")
        g.add((factory_node, ONT.hasMachine, mnode)); g.add((mnode, ONT.belongsToFactory, factory_node))
        add_dt(g, mnode, "machineID", mid, XSD.string)
        info = machine_info.get(mid, {})
        model_name = str(info.get("model", "unknown_model"))
        model_node = uri(factory_id, "machine-model", model_name)
        g.add((model_node, RDF.type, ONT.MachineModel)); add_label(g, model_node, model_name)
        add_dt(g, model_node, "modelName", model_name, XSD.string)
        g.add((mnode, ONT.usesModel, model_node))
        add_dt(g, mnode, "modelName", model_name, XSD.string)
        age = info.get("age", np.nan)
        if pd.notna(age): add_dt(g, mnode, "machineAge", float(age), XSD.float)

    sub = daily[daily["client_id"] == factory_id]
    for _, row in sub.iterrows():
        mid = row["machineID"]
        mnode = uri(factory_id, "machine", mid)
        d = pd.Timestamp(row["day"])
        dstr = d.strftime("%Y-%m-%d")
        node = uri(factory_id, "telemetry-daily", f"{mid}/{dstr}")
        g.add((node, RDF.type, ONT.TelemetryDailySummary)); add_label(g, node, f"Telemetry daily summary for machine {mid} on {dstr}")
        g.add((mnode, ONT.hasDailyTelemetry, node))
        add_dt(g, node, "dailyDate", dstr, XSD.date)
        add_dt(g, node, "nSamples", int(row["n_samples"]), XSD.integer)
        for sensor in SENSORS:
            for stat in ["mean", "std", "min", "max"]:
                prop = f"{sensor}{stat.title()}"
                add_dt(g, node, prop, float(row[f"{sensor}_{stat}"]), XSD.float)

    local_errors = errors[errors["client_id"] == factory_id]
    for _, row in local_errors.iterrows():
        mid, code, ts = row["machineID"], str(row["errorID"]), pd.Timestamp(row["datetime"])
        event_key = f"{mid}/{ts.isoformat()}/{code}"
        ev = uri(factory_id, "error-occurrence", event_key)
        mnode = uri(factory_id, "machine", mid)
        code_node = uri(factory_id, "error-code", code)
        g.add((ev, RDF.type, ONT.ErrorOccurrence)); add_label(g, ev, f"Error {code} on machine {mid} at {ts.isoformat()}")
        g.add((mnode, ONT.hasErrorOccurrence, ev)); g.add((ev, ONT.hasErrorCode, code_node))
        g.add((code_node, RDF.type, ONT.ErrorCode)); add_label(g, code_node, code)
        add_dt(g, code_node, "errorCode", code, XSD.string)
        g.add((ev, ONT.occurredAt, event_datetime(ts)))

    local_maint = maintenance[maintenance["client_id"] == factory_id]
    for _, row in local_maint.iterrows():
        mid, comp, ts = row["machineID"], str(row["comp"]), pd.Timestamp(row["datetime"])
        ev = uri(factory_id, "maintenance-event", f"{mid}/{ts.isoformat()}/{comp}")
        mnode = uri(factory_id, "machine", mid)
        cnode = uri(factory_id, "component", comp)
        g.add((ev, RDF.type, ONT.MaintenanceEvent)); add_label(g, ev, f"Maintenance on machine {mid}, component {comp}, at {ts.isoformat()}")
        g.add((mnode, ONT.hasMaintenanceEvent, ev)); g.add((ev, ONT.targetsComponent, cnode))
        g.add((cnode, RDF.type, ONT.Component)); add_label(g, cnode, comp); add_dt(g, cnode, "componentName", comp, XSD.string)
        g.add((ev, ONT.occurredAt, event_datetime(ts)))

    local_fail = failures[failures["client_id"] == factory_id]
    for _, row in local_fail.iterrows():
        mid, comp, ts = row["machineID"], str(row["failure_type"]), pd.Timestamp(row["datetime"])
        ev = uri(factory_id, "failure-event", f"{mid}/{ts.isoformat()}/{comp}")
        mnode = uri(factory_id, "machine", mid)
        cnode = uri(factory_id, "component", comp)
        g.add((ev, RDF.type, ONT.FailureEvent)); add_label(g, ev, f"Observed failure on machine {mid}, label {comp}, at {ts.isoformat()}")
        g.add((mnode, ONT.hasFailureEvent, ev)); g.add((ev, ONT.failureOfComponent, cnode))
        g.add((cnode, RDF.type, ONT.Component)); add_label(g, cnode, comp); add_dt(g, cnode, "componentName", comp, XSD.string)
        g.add((ev, ONT.occurredAt, event_datetime(ts)))

    out_path = OUT_DIR / "graphs" / f"{factory_id}.ttl"
    g.serialize(destination=str(out_path), format="turtle")
    local_graphs[factory_id] = g
    graph_stats.append({
        "factory": factory_id,
        "machines": len(factory_machines),
        "daily_telemetry_nodes": len(sub),
        "error_occurrences": len(local_errors),
        "maintenance_events": len(local_maint),
        "failure_records": len(local_fail),
        "rdf_triples": len(g),
        "ttl_size_mb": round(out_path.stat().st_size / (1024**2), 3),
        "graph_path": str(out_path),
    })

stats_df = pd.DataFrame(graph_stats)
stats_df.to_csv(OUT_DIR / "reports" / "factory_graph_statistics.csv", index=False)
print(stats_df.to_string(index=False))

In [ ]:
# Validation checks: structural and partition checks, not model-performance claims.
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "status": "PASS" if bool(ok) else "FAIL", "detail": str(detail)})
    print(f"[{ 'PASS' if ok else 'FAIL' }] {name}: {detail}")

check("V1 source dataset is Azure tables only", all("PdM_" in str(p) or p.name.lower() in {"telemetry.csv", "errors.csv", "maintenance.csv", "maint.csv", "failures.csv", "machines.csv"} for p in PATHS.values()),
      "No NASA C-MAPSS input is loaded in this notebook.")
check("V2 five expected factory partitions are present", set(local_graphs) == set(EXPECTED_FACTORIES), str(sorted(local_graphs)))
check("V3 each source machine maps to exactly one factory", bool((ids_per_machine == 1).all()) and mapping_df["machineID"].nunique() == len(mapping_df), f"mapped machines={mapping_df['machineID'].nunique()}")
check("V4 machine coverage matches the source inventory", set(machines["machineID"]) <= set(factory_map), f"source inventory machines={machines['machineID'].nunique()}")
check("V5 shared ontology file exists", ontology_path.exists() and len(ontology) > 0, f"ontology triples={len(ontology)}")
check("V6 all local graph files exist", all((OUT_DIR / "graphs" / f"{f}.ttl").exists() for f in EXPECTED_FACTORIES), "one RDF/Turtle file per factory")

for f in EXPECTED_FACTORIES:
    g = local_graphs[f]
    expected = {m for m, assigned in factory_map.items() if assigned == f}
    actual = {str(g.value(node, ONT.machineID)) for node in g.subjects(RDF.type, ONT.Machine) if g.value(node, ONT.machineID) is not None}
    check(f"V7 {f} machine isolation", actual == expected, f"expected={len(expected)} actual={len(actual)}")
    error_nodes = set(g.subjects(RDF.type, ONT.ErrorOccurrence))
    maint_nodes = set(g.subjects(RDF.type, ONT.MaintenanceEvent))
    failure_nodes = set(g.subjects(RDF.type, ONT.FailureEvent))
    bad_err = [ev for ev in error_nodes if not list(g.subjects(ONT.hasErrorOccurrence, ev)) or not list(g.objects(ev, ONT.hasErrorCode)) or not list(g.objects(ev, ONT.occurredAt))]
    bad_maint = [ev for ev in maint_nodes if not list(g.subjects(ONT.hasMaintenanceEvent, ev)) or not list(g.objects(ev, ONT.targetsComponent)) or not list(g.objects(ev, ONT.occurredAt))]
    bad_fail = [ev for ev in failure_nodes if not list(g.subjects(ONT.hasFailureEvent, ev)) or not list(g.objects(ev, ONT.failureOfComponent)) or not list(g.objects(ev, ONT.occurredAt))]
    check(f"V8 {f} error events linked", not bad_err, f"events={len(error_nodes)} broken={len(bad_err)}")
    check(f"V9 {f} maintenance events linked", not bad_maint, f"events={len(maint_nodes)} broken={len(bad_maint)}")
    check(f"V10 {f} failure records linked", not bad_fail, f"events={len(failure_nodes)} broken={len(bad_fail)}")
    tele_nodes = set(g.subjects(RDF.type, ONT.TelemetryDailySummary))
    bad_tele = [n for n in tele_nodes if not list(g.subjects(ONT.hasDailyTelemetry, n)) or not list(g.objects(n, ONT.dailyDate))]
    check(f"V11 {f} daily telemetry linked", not bad_tele, f"summaries={len(tele_nodes)} broken={len(bad_tele)}")
    # No causal relation names are declared in the shared ontology.
    causal_props = [p for p in ontology.subjects(RDF.type, OWL.ObjectProperty) if any(t in str(p).lower() for t in ["causes", "causedby", "leadsTo".lower()])]
    check(f"V12 {f} ontology avoids unsupported causal edges", len(causal_props) == 0, "relationships express observations/records only")

checks_df = pd.DataFrame(checks)
checks_df.to_csv(OUT_DIR / "reports" / "phase4_kg_validation_checks.csv", index=False)
failed = checks_df[checks_df["status"] != "PASS"]
if len(failed):
    display(failed)
    raise AssertionError(f"Phase 4 has {len(failed)} failed validation checks; do not treat the graph as closed.")
print(f"\nAll {len(checks_df)} structural checks passed.")

In [ ]:
# Temporal retrieval smoke test: only facts at or before `as_of` may be returned.
def _parse_rdf_datetime(value):
    if value is None:
        return None
    try: return pd.Timestamp(str(value))
    except Exception: return None

def retrieve_as_of(factory_id, machine_id, as_of, lookback_days=30):
    """Small evidence retriever for smoke tests and the Phase 5 notebook.
    Returns event records and daily telemetry strictly no later than as_of.
    """
    g = local_graphs[factory_id]
    as_of = pd.Timestamp(as_of)
    cutoff = as_of - pd.Timedelta(days=int(lookback_days))
    mnode = uri(factory_id, "machine", normalize_id(machine_id))
    if (mnode, RDF.type, ONT.Machine) not in g:
        raise KeyError(f"Machine {machine_id} not present in {factory_id}")
    out = []
    for pred, kind, related_pred in [
        (ONT.hasErrorOccurrence, "error", ONT.hasErrorCode),
        (ONT.hasMaintenanceEvent, "maintenance", ONT.targetsComponent),
        (ONT.hasFailureEvent, "failure", ONT.failureOfComponent),
    ]:
        for ev in g.objects(mnode, pred):
            ts = _parse_rdf_datetime(g.value(ev, ONT.occurredAt))
            if ts is not None and cutoff <= ts <= as_of:
                related = g.value(ev, related_pred)
                label = str(g.value(related, ONT.errorCode) or g.value(related, ONT.componentName) or g.value(related, RDFS.label) or "unknown")
                out.append({"kind": kind, "timestamp": ts.isoformat(), "observed_value": label})
    for tele in g.objects(mnode, ONT.hasDailyTelemetry):
        d = _parse_rdf_datetime(g.value(tele, ONT.dailyDate))
        if d is not None and cutoff.normalize() <= d.normalize() < as_of.normalize():
            out.append({"kind": "daily_telemetry", "timestamp": d.strftime("%Y-%m-%d"),
                        "volt_mean": float(g.value(tele, ONT.voltMean)),
                        "rotate_mean": float(g.value(tele, ONT.rotateMean)),
                        "pressure_mean": float(g.value(tele, ONT.pressureMean)),
                        "vibration_mean": float(g.value(tele, ONT.vibrationMean))})
    out.sort(key=lambda r: r["timestamp"])
    return out

# Test using the most recent observed telemetry date in the graph.
smoke_factory = "factory_5"
smoke_mid = next(m for m, f in factory_map.items() if f == smoke_factory)
smoke_date = daily[daily["machineID"].eq(smoke_mid)]["day"].max()
smoke = retrieve_as_of(smoke_factory, smoke_mid, smoke_date, lookback_days=30)
smoke_df = pd.DataFrame(smoke)
if len(smoke_df):
    event_times = pd.to_datetime(smoke_df.loc[smoke_df["kind"] != "daily_telemetry", "timestamp"], errors="coerce")
    assert (event_times.dropna() <= pd.Timestamp(smoke_date)).all(), "Retriever returned an event after as_of."
print("As-of retrieval smoke test:", {"factory": smoke_factory, "machine": smoke_mid, "as_of": str(smoke_date.date()), "facts_returned": len(smoke)})
display(smoke_df.tail(12))

In [ ]:
# Final human-readable report and complete status artifact.
summary = {
    "phase": "Phase 4 — Factory-Local Maintenance Knowledge Graphs v1",
    "source_scope": "Azure Predictive Maintenance tables only; NASA C-MAPSS excluded by project scope decision",
    "normalization_and_factory_mapping": "existing Phase 1 fedkg_data metadata; no new partition assignment",
    "factories": EXPECTED_FACTORIES,
    "machine_count": int(mapping_df["machineID"].nunique()),
    "sensor_fields": SENSORS,
    "daily_telemetry_summary_count": int(len(daily)),
    "error_occurrences_after_dedup": int(len(errors)),
    "maintenance_events_after_dedup": int(len(maintenance)),
    "failure_records_after_dedup": int(len(failures)),
    "shared_ontology_path": str(ontology_path),
    "factory_graphs": graph_stats,
    "validation_checks_total": int(len(checks_df)),
    "validation_checks_passed": int((checks_df["status"] == "PASS").sum()),
    "validation_checks_failed": int((checks_df["status"] == "FAIL").sum()),
    "interpretation_limitations": [
        "Relations reflect records and observed associations, not proven causes.",
        "The data are processed in one notebook process; this is not evidence of physical factory data isolation.",
        "Each explanation must retrieve only events and telemetry at or before its prediction timestamp.",
        "Maintenance component labels and failure labels are taken from the dataset; no extra engineering failure-mode ontology is invented."
    ],
    "decision": "PASS — local graph artifacts created and structural checks passed" if (checks_df["status"] == "PASS").all() else "FAIL — inspect validation report"
}
(OUT_DIR / "reports" / "phase4_kg_build_report.json").write_text(json.dumps(summary, indent=2, default=str))
md = [
    "# Phase 4 — Factory-Local Maintenance Knowledge Graphs v1", "",
    f"- Decision: **{summary['decision']}**",
    f"- Machines: **{summary['machine_count']}** across 5 local graphs.",
    f"- Daily telemetry summaries: **{summary['daily_telemetry_summary_count']:,}**.",
    f"- Error occurrences: **{summary['error_occurrences_after_dedup']:,}**; maintenance events: **{summary['maintenance_events_after_dedup']:,}; failure records: **{summary['failure_records_after_dedup']:,}**.",
    f"- Structural checks: **{summary['validation_checks_passed']}/{summary['validation_checks_total']} PASS**.",
    "", "## Factory graph statistics", "", "| Factory | Machines | Daily telemetry | Errors | Maintenance | Failures | RDF triples |", "|---|---:|---:|---:|---:|---:|---:|"
]
for r in graph_stats:
    md.append(f"| {r['factory']} | {r['machines']} | {r['daily_telemetry_nodes']:,} | {r['error_occurrences']:,} | {r['maintenance_events']:,} | {r['failure_records']:,} | {r['rdf_triples']:,} |")
md += ["", "## Limitations"] + [f"- {x}" for x in summary["interpretation_limitations"]]
md += ["", "## Artifact paths", f"- Shared ontology: `{ontology_path}`"] + [f"- `{r['graph_path']}`" for r in graph_stats]
(OUT_DIR / "reports" / "phase4_kg_build_report.md").write_text("\n".join(md))
print("\n".join(md[:10]))
print("\nReports saved to:", OUT_DIR / "reports")

## Phase 4 completion boundary

This notebook creates the graph assets; it does not yet claim LLM explanation quality, retrieval recall, physical privacy isolation, or prediction improvement. Phase 5 should read these local graph artifacts, enforce timestamp-aware retrieval, and compare the same prediction cases with versus without graph evidence.